# Ira Fine-Tuning on Google Colab

Fine-tune Llama 3.1 8B on BEAM/SCORE content with Buffett, Robbins, and Singer wisdom.

**Requirements:**
- Google Colab Pro (recommended) or free tier with patience
- HuggingFace account and token
- Training data uploaded to this notebook

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets accelerate peft trl bitsandbytes huggingface_hub

In [ ]:
# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Login to HuggingFace using Colab Secrets (secure - token never exposed)
# 
# SETUP (one time):
# 1. Click the 🔑 key icon in the left sidebar
# 2. Add a new secret named: HF_TOKEN
# 3. Paste your "Unchained" token value (starts with hf_)
# 4. Toggle "Notebook access" ON
#
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("✓ Logged in to HuggingFace via Colab Secrets!")
except Exception as e:
    print("⚠ Secret not found. Using manual login...")
    login()  # Fallback to manual prompt

In [ ]:
# Upload your training data
from google.colab import files
print("Upload train.jsonl and eval.jsonl from training/data/")
uploaded = files.upload()

In [ ]:
# Load training data
from datasets import load_dataset

dataset = load_dataset('json', data_files={
    'train': 'train.jsonl',
    'eval': 'eval.jsonl'
})

print(f"Train: {len(dataset['train'])} examples")
print(f"Eval: {len(dataset['eval'])} examples")

In [ ]:
# Model configuration
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
OUTPUT_REPO = "BEAMcollective/ira-beam-assistant"

# Quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Model loaded!")

In [ ]:
# LoRA configuration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Format dataset for training
def format_chat(example):
    messages = example.get('messages', [])
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {'text': text}

formatted_dataset = dataset.map(format_chat)
print("Dataset formatted!")

In [ ]:
# Training configuration
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./ira-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    weight_decay=0.001,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    evaluation_strategy="steps",
    fp16=True,
    gradient_checkpointing=True,
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    push_to_hub=True,
    hub_model_id=OUTPUT_REPO,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset['train'],
    eval_dataset=formatted_dataset['eval'],
    tokenizer=tokenizer,
    dataset_text_field='text',
    max_seq_length=2048,
    packing=False,
)

print("Trainer ready!")

In [ ]:
# Start training!
print("Starting training... This will take 1-3 hours on Colab Pro.")
trainer.train()

In [ ]:
# Save and push to HuggingFace
trainer.save_model("./ira-final")
trainer.push_to_hub()

print(f"\nModel pushed to: huggingface.co/{OUTPUT_REPO}")
print("\nUpdate your .env.local with:")
print(f"HF_INFERENCE_MODEL={OUTPUT_REPO}")

In [ ]:
# Test Ira!
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

messages = [
    {"role": "system", "content": "You are Ira, the wise AI mentor for BEAM."},
    {"role": "user", "content": "I'm scared to start my business. What should I do?"}
]

response = pipe(messages, max_new_tokens=200)
print(response[0]['generated_text'][-1]['content'])